# Magnetic Field of Earth

### imports

In [6]:
!pip3 install pyIGRF
!pip3 install numpy

    sqlalchemy (>=1.4.1<2.0.0)
               ~~~~~~~~^

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
    sqlalchemy (>=1.4.1<2.0.0)
               ~~~~~~~~^

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip


In [1]:
import pyIGRF
import numpy as np
import math

In [2]:
def earth_magnetic_field(lat, lon, alt, year):
    Bx, By, Bz, Bh, Bt, Dec, Inc = pyIGRF.igrf_value(lat, lon, alt, year)
    print(f"North Component (X): {Bx} nT")
    print(f"East Component (Y): {By} nT")
    print(f"Down Component (Z): {Bz} nT")
    print(f"Total Field (Bt): {Bt} nT")
    print(f"Horizontal Intensity (Bh): {Bh} nT")
    print(f"Declination (D): {Dec}°")
    print(f"Inclination (I): {Inc}°")

# Converting Coordinates from North East Down to Latitude, Longitude, Altitude

In [3]:
# WGS84 ellipsoid parameters
a = 6378137.0  # semi-major axis (meters)
f = 1 / 298.257223563  # flattening
e = math.sqrt(2 * f - f ** 2)  # eccentricity
e2 = e**2

def lla_to_ecef(lat, lon, h):
    """Convert LLA (Latitude, Longitude, Altitude) to ECEF (X, Y, Z)."""
    lat_rad = math.radians(lat)
    lon_rad = math.radians(lon)
    
    N = a / math.sqrt(1 - e2 * math.sin(lat_rad) ** 2)
    
    X = (N + h) * math.cos(lat_rad) * math.cos(lon_rad)
    Y = (N + h) * math.cos(lat_rad) * math.sin(lon_rad)
    Z = (N * (1 - e2) + h) * math.sin(lat_rad)
    
    return X, Y, Z

def ned_to_ecef(n, e, d, lat_ref, lon_ref, h_ref):
    """Convert NED (North, East, Down) to ECEF (X, Y, Z)."""
    # Get ECEF coordinates of the reference point
    Xr, Yr, Zr = lla_to_ecef(lat_ref, lon_ref, h_ref)
    
    lat_ref_rad = math.radians(lat_ref)
    lon_ref_rad = math.radians(lon_ref)
    
    # Rotation matrix from NED to ECEF
    R = np.array([
        [-math.sin(lat_ref_rad) * math.cos(lon_ref_rad), -math.sin(lon_ref_rad), -math.cos(lat_ref_rad) * math.cos(lon_ref_rad)],
        [-math.sin(lat_ref_rad) * math.sin(lon_ref_rad), math.cos(lon_ref_rad), -math.cos(lat_ref_rad) * math.sin(lon_ref_rad)],
        [math.cos(lat_ref_rad), 0, -math.sin(lat_ref_rad)]
    ])
    
    # Convert NED to ECEF
    ned_vector = np.array([n, e, d])
    ecef_vector = R.T @ ned_vector  # Apply rotation
    
    # New ECEF position
    X = Xr + ecef_vector[0]
    Y = Yr + ecef_vector[1]
    Z = Zr + ecef_vector[2]
    
    return X, Y, Z

def ecef_to_geodetic(X, Y, Z):
    """Convert ECEF (X, Y, Z) to LLA (Latitude, Longitude, Altitude)."""
    lon = np.arctan2(Y, X)
    p = np.sqrt(X ** 2 + Y ** 2)
    lat = np.arctan2(Z, p * (1 - e2))  # initial guess
    
    N = a / math.sqrt(1 - e2 * np.sin(lat) ** 2)
    h = p / np.cos(lat) - N
    
    for _ in range(5):
        N = a / np.sqrt(1 - e2 * np.sin(lat)**2)
        h = p / np.cos(lat) - N
        lat = np.arctan2(Z, p * (1 - e2 * N / (N + h)))
    
    lat = np.degrees(lat)
    lon = np.degrees(lon)
    
    return lat, lon, h

def ned_to_lla(n, e, d, lat_ref, lon_ref, h_ref):
    """Convert NED coordinates to LLA using reference LLA."""
    X, Y, Z = ned_to_ecef(n, e, d, lat_ref, lon_ref, h_ref)
    return ecef_to_geodetic(X, Y, Z)

In [4]:
lat_ref = 35.3533333   # degrees
lon_ref = -117.8071667 # degrees
h_ref = 622            # meters (altitude)

print(ned_to_lla(120, 320, 20, lat_ref, lon_ref, h_ref))

(35.35559859390918, -117.80487779857965, 725.582362877205)


# Barometer

In [5]:
P_0 = 101.325  # kPa (standard atmospheric pressure at sea level)
T_0 = 288.15  # K (standard temperature at sea level)
g = 9.80665 # m/s^2
L = 0.0065 # K/m in the troposphere
R = 8.3144598 # universal gas constant
M = 0.0289644 # molar mass of Earth's air

def pressureToAltitude(p):
    return (T_0/L)*(1-(p/P_0)**((R*L)/(g*M)))

def altitudeToPressure(h):
    return P_0 * (1 - L * h / T_0) ** (g * M / (R * L))

# exact temperature
def altitudeTemperatureToPressure(h, t):
    return P_0*math.exp(-g*M*h/(R*t))


In [10]:
print(altitudeTemperatureToPressure(20000,288.15))

9.460866433468707


In [14]:
import math

def standard_atmosphere_pressure(h):
    """
    Returns the standard-atmosphere pressure (Pa) at geometric altitude h (m),
    using the 1976 US Standard Atmosphere piecewise definitions up to 20 km.
    
    Valid up to 20,000 m. Above 20 km, further segments of the standard
    atmosphere apply, but are not shown here for brevity.
    """
    
    # --- Constants ---
    g0   = 9.80665       # m/s^2  (sea-level gravitational accel)
    R    = 8.31432       # J/(mol·K) universal gas constant
    M    = 0.0289644     # kg/mol  (mean molar mass of air)
    
    # --- Troposphere (0 to 11 km) ---
    #   T(h) = T0 - L*h, where L=0.0065 K/m, T0=288.15 K at h=0
    #   P(h) = P0 * [T(h)/T0]^(g0·M / (R·L))
    #
    #   at h=0, T0=288.15 K, P0=101325 Pa
    
    # Base conditions for start of troposphere layer
    h0_trop = 0.0
    T0_trop = 288.15      # K
    P0_trop = 101325.0    # Pa
    L_trop  = 0.0065      # K/m
    
    # Height of troposphere top
    h_trop_top = 11000.0  # m
    
    # Temperature at top of troposphere:
    T_trop_top = T0_trop - L_trop * (h_trop_top - h0_trop)  # 288.15 - 0.0065*11000 = 216.65 K
    
    # Pressure at top of troposphere:
    exponent_trop = (g0 * M) / (R * L_trop)
    P_trop_top = P0_trop * (T_trop_top / T0_trop)**exponent_trop
    
    # --- Lower Stratosphere (11 km to 20 km) ---
    #   T is constant = 216.65 K
    #   P(h) = P(11km) * exp[ -g0·M (h - 11000) / (R·T_trop_top) ]
    
    h_strat_top = 20000.0  # m
    T_strat      = T_trop_top  # 216.65 K
    
    def pressure_in_troposphere(h):
        # Linear temperature drop from 0 to 11 km
        T_h = T0_trop - L_trop * (h - h0_trop)
        return P0_trop * (T_h / T0_trop)**exponent_trop
    
    def pressure_in_stratosphere(h):
        # Isothermal from 11 km to 20 km
        return P_trop_top * math.exp(
            - (g0 * M * (h - h_trop_top)) / (R * T_strat)
        )
    
    # Now piecewise evaluate
    if h <= h_trop_top:
        # Entirely within troposphere
        return pressure_in_troposphere(h)
    else:
        # Above 11 km but below (or at) 20 km
        if h > h_strat_top:
            raise ValueError("This code snippet only covers up to 20 km!")
        return pressure_in_stratosphere(h)

# --- Example usage ---
if __name__ == "__main__":
    alt = 18000.0  # m
    p_at_20km = standard_atmosphere_pressure(alt)
    print(f"Standard-atmosphere pressure at {alt} m ≈ {p_at_20km:.2f} Pa")


Standard-atmosphere pressure at 18000.0 m ≈ 7504.84 Pa
